In [15]:
import pandas as pd
import numpy as np
from scipy import stats

In [9]:
states = pd.read_csv("data-oKFfw.csv")
states.head(5)

,SUMLEV,REGION,DIVISION,STATE,NAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,...,DOMESTICMIG2022,DOMESTICMIG2023,DOMESTICMIG2024,DOMESTICMIG2025,RATEDOMESTICMIG2021,RATEDOMESTICMIG2022,RATEDOMESTICMIG2023,RATEDOMESTICMIG2024,RATEDOMESTICMIG2025,rank
0,40,South,South Atlantic,45,South Carolina,5118250,5131992,5194346,5288957,5390798,...,83341,79536,66367,66622,13.2,15.9,14.9,12.2,12.05,1st
1,40,West,Mountain,16,Idaho,1839123,1849328,1904855,1942951,1970497,...,28019,14713,15975,19915,27.9,14.6,7.5,8.0,9.88,2nd
2,40,South,South Atlantic,37,North Carolina,10441392,10450215,10565503,10705768,10871849,...,98454,98929,83059,84064,9.9,9.3,9.2,7.6,7.56,3rd
3,40,South,South Atlantic,10,Delaware,989950,991890,1005130,1020279,1035354,...,12530,9872,8040,6855,13.7,12.4,9.6,7.7,6.50,4th
4,40,South,East South Central,47,Tennessee,6912319,6927736,6966687,7063325,7153029,...,82316,60397,46496,42389,6.7,11.7,8.5,6.5,5.82,5th


Descriptive Analysis

In [10]:
states['pop_growth_pct'] = (states['POPESTIMATE2025'] - states['POPESTIMATE2020']) / states['POPESTIMATE2020'] * 100
states['avg_mig_rate']   = states[['RATEDOMESTICMIG2021','RATEDOMESTICMIG2022',
                            'RATEDOMESTICMIG2023','RATEDOMESTICMIG2024',
                            'RATEDOMESTICMIG2025']].mean(axis=1)
states['total_net_mig']  = states[['DOMESTICMIG2021','DOMESTICMIG2022',
                            'DOMESTICMIG2023','DOMESTICMIG2024',
                            'DOMESTICMIG2025']].sum(axis=1)
states['mig_trend']      = states['RATEDOMESTICMIG2025'] - states['RATEDOMESTICMIG2021']

In [11]:
rate_cols = ['RATEDOMESTICMIG2021','RATEDOMESTICMIG2022',
             'RATEDOMESTICMIG2023','RATEDOMESTICMIG2024','RATEDOMESTICMIG2025']
 
print("\n--- Migration Rate Descriptive Stats (per 1,000 population) ---")
print(states[rate_cols].describe().round(3))


--- Migration Rate Descriptive Stats (per 1,000 population) ---
       RATEDOMESTICMIG2021  RATEDOMESTICMIG2022  RATEDOMESTICMIG2023  \
count               51.000               51.000               51.000   
mean                 2.325                1.061                0.890   
std                  7.881                7.515                5.274   
min                -16.300              -14.400               -9.000   
25%                 -2.800               -3.100               -2.000   
50%                  1.700                0.100                0.400   
75%                  6.800                6.750                4.850   
max                 27.900               15.900               14.900   

       RATEDOMESTICMIG2024  RATEDOMESTICMIG2025  
count               51.000               51.000  
mean                 0.894                0.976  
std                  3.847                4.048  
min                 -6.200               -6.880  
25%                 -0.900          

In [12]:
print("\n--- Population Growth % (2020→2025) by Region ---")
print(states.groupby('REGION')['pop_growth_pct'].agg(['mean','std','min','max']).round(2))


--- Population Growth % (2020→2025) by Region ---
           mean   std   min   max
REGION                           
Midwest    1.85  1.46 -0.60  5.33
Northeast  1.84  1.48 -0.60  3.69
South      4.03  3.23 -1.42  8.67
West       3.35  3.38 -1.26  9.76


In [13]:
print("\n--- Top 10 Gainers by Total Net Migration (2021-2025) ---")
top10 = states.nlargest(10, 'total_net_mig')[['NAME','total_net_mig','avg_mig_rate']]
print(top10.to_string(index=False))


--- Top 10 Gainers by Total Net Migration (2021-2025) ---
          NAME  total_net_mig  avg_mig_rate
       Florida         828686         7.472
         Texas         760233         5.048
North Carolina         468133         8.712
South Carolina         364272        13.650
     Tennessee         277999         7.844
       Arizona         252762         6.860
       Georgia         221265         4.046
       Alabama         131598         5.162
         Idaho         130972        13.576
      Oklahoma         103396         5.126


In [14]:
print("\n--- Top 10 Losers by Total Net Migration (2021-2025) ---")
bot10 = states.nsmallest(10, 'total_net_mig')[['NAME','total_net_mig','avg_mig_rate']]
print(bot10.to_string(index=False))


--- Top 10 Losers by Total Net Migration (2021-2025) ---
         NAME  total_net_mig  avg_mig_rate
   California       -1629468        -8.324
     New York       -1045769       -10.516
     Illinois        -431149        -6.790
   New Jersey        -211131        -4.526
Massachusetts        -145282        -4.094
    Louisiana        -138644        -6.004
     Maryland        -129734        -4.188
     Michigan         -60803        -1.204
       Hawaii         -54537        -7.578
 Pennsylvania         -50746        -0.784


Hypothesis Testing 

Hypothesis 1: Southern states have higher migration rates than non-Southern states
HO: Mean migration rate (2025) is equal for South vs non-South
H1: Southern states have higher migration rate than non-South

In [16]:
south = states[states['REGION'] == 'South']['RATEDOMESTICMIG2025']
non_south = states[states['REGION'] != 'South']['RATEDOMESTICMIG2025']
t_stat, p_val = stats.ttest_ind(south, non_south, alternative='greater')
print(f"  South mean rate: {south.mean():.2f} | Non-South mean: {non_south.mean():.2f}")
print(f"  t-statistic: {t_stat:.3f}  |  p-value: {p_val:.4f}")
print(f"  Result: {'REJECT H0' if p_val < 0.05 else 'FAIL TO REJECT H0'} (α=0.05)")
print(f"  Interpretation: {'Southern states DO have significantly higher migration rates.' if p_val < 0.05 else 'No significant difference.'}")

  South mean rate: 2.63 | Non-South mean: 0.15
  t-statistic: 2.140  |  p-value: 0.0187
  Result: REJECT H0 (α=0.05)
  Interpretation: Southern states DO have significantly higher migration rates.
